# Preprocessing Experiments
Resize and normalization comparisons for chest X-Ray images.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from preprocessing.dataset_loader import DatasetLoader
from preprocessing.preprocessor import Preprocessor

logging.basicConfig(level=logging.INFO)
PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'output' / '01_preprocessing'

In [ ]:
# Load dataset
loader = DatasetLoader()
images, labels, paths = loader.load_original()
print(f'Loaded {len(images)} original images')
print(f'Unique labels: {loader.get_unique_labels()}')
print(f'Sample image shape: {images[0].shape}, dtype: {images[0].dtype}')

In [ ]:
# Visualize 5 sample images
n_samples = min(5, len(images))
fig, axes = plt.subplots(1, n_samples, figsize=(18, 4))
for i, ax in enumerate(axes):
    ax.imshow(images[i], cmap='gray')
    ax.set_title(f'{labels[i]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Original Images', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=150)
plt.show()

In [ ]:
# Resize comparison
prep = Preprocessor()
sample = images[0]
size_variants = prep.compare_sizes(sample)

fig, axes = plt.subplots(1, len(size_variants), figsize=(14, 4))
for ax, (name, img) in zip(axes, size_variants.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{name}\n{img.shape}', fontsize=9)
    ax.axis('off')
plt.suptitle('Resize Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'resize_comparison.png', dpi=150)
plt.show()

In [ ]:
# Normalization comparison with histograms
norm_variants = prep.compare_normalizations(sample)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col_idx, (name, img) in enumerate(norm_variants.items()):
    axes[0, col_idx].imshow(img, cmap='gray')
    axes[0, col_idx].set_title(f'{name}', fontsize=10)
    axes[0, col_idx].axis('off')

    axes[1, col_idx].hist(img.ravel(), bins=64, color='steelblue', alpha=0.7)
    axes[1, col_idx].set_title(f'{name} histogram')
    axes[1, col_idx].set_xlabel('Pixel value')
    axes[1, col_idx].set_ylabel('Frequency')

plt.suptitle('Normalization Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'normalization_comparison.png', dpi=150)
plt.show()

In [ ]:
# Batch process all original images
from pathlib import Path
orig_paths = [Path(p) for p in paths]

saved_minmax = prep.batch_process(
    orig_paths, OUTPUT_DIR,
    target_size=(224, 224), norm_method='minmax',
    subfolder='normalized'
)
print(f'Saved {len(saved_minmax)} minmax-normalized images')

saved_resized = prep.batch_process(
    orig_paths, OUTPUT_DIR,
    target_size=(224, 224), norm_method='minmax',
    subfolder='resized'
)
print(f'Saved {len(saved_resized)} resized images')

In [ ]:
# Label distribution
import pandas as pd
label_df = loader.get_label_dataframe('original')
counts = label_df['label'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Class Distribution — Original Balanced Dataset')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'label_distribution.png', dpi=150)
plt.show()
print(counts)